In [1]:
# UKB PROJECTION + REGRESSION SCRIPT (UPDATED)
# 1) Loads the saved FINAL RF models + training metadata
# 2) Builds UKB X with EXACT training columns (missing cols -> NaN)
# 3) Coerces to numeric (strings -> NaN)
# 4) Uses the *trained* SimpleImputer inside each saved RF pipeline:
#    - If a whole column is missing in UKB (e.g., dog/cat), it becomes NaN,
#      and is imputed to the AGP median (constant for all UKB participants).
#    - If some entries are missing, those entries get AGP-median imputation.
# 5) Predicts PC_hat for UKB
# 6) Runs OLS regression: cognition ~ PC_hat (+ optional covariates)|

import os
import json
import joblib
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests


# 0) Config
model_dir = "./RF_PCA_Trained_Models/"  # where the saved final_*.pkl and final_training_metadata.json
ukb_metadata_path = "variable_mapping/ukb_as_agp_metadata.csv"
ukb_index_col = "sample_name"

# outcome variable
cognition_col = "fluid_intelligence_score"
# cognition_col = "mean_match_rt_ms"

reg_covars = [
    "age_corrected",
    "race",
    #"country_of_birth"
]

out_dir = "./hongrui_result/UKB_RF_PCA_Projection/"
os.makedirs(out_dir, exist_ok=True)

# 1) LOAD SAVED MODELS + TRAINING METADATA
final_rf_models = joblib.load(os.path.join(model_dir, "final_rf_models.pkl"))

with open(os.path.join(model_dir, "final_training_metadata.json"), "r") as f:
    train_meta = json.load(f)

train_cols = train_meta["numeric_cols_used_as_X"]
n_pcs_to_predict = int(train_meta["n_pcs_to_predict"])

print(f"Loaded {len(final_rf_models)} RF models.")
print(f"Training expects {len(train_cols)} numeric metadata columns.")
print(f"Will predict {n_pcs_to_predict} PCs.")

# 2) LOAD UKB METADATA (AGP-LIKE)
ukb_df = pd.read_csv(ukb_metadata_path, index_col=ukb_index_col)
ukb_df["age_corrected"] = pd.to_numeric(ukb_df["age_corrected"], errors="coerce")
print(f"Loaded UKB-like metadata: {ukb_df.shape[0]} rows, {ukb_df.shape[1]} columns.")

if cognition_col not in ukb_df.columns:
    raise ValueError(f"cognition_col='{cognition_col}' not found. Available cols include: {list(ukb_df.columns)[:30]} ...")

missing_covars = [c for c in reg_covars if c not in ukb_df.columns]
if missing_covars:
    raise ValueError(f"Missing regression covariates in UKB dataframe: {missing_covars}")

Loaded 10 RF models.
Training expects 29 numeric metadata columns.
Will predict 10 PCs.
Loaded UKB-like metadata: 502244 rows, 48 columns.


In [2]:
# Build covariates ONCE (consistent across OLS/baseline/permutation/logit)

cov_num = ["age_corrected"]                 # continuous
cov_cat = [
    "race", 
    # "country_of_birth"
]# categorical

# build cov_df
cov_df_num = ukb_df[cov_num].apply(pd.to_numeric, errors="coerce")
cov_df_cat = pd.get_dummies(
    ukb_df[cov_cat].astype("string").fillna("MISSING"),
    prefix=cov_cat,
    drop_first=True,
    dtype=float
)

cov_df = pd.concat([cov_df_num, cov_df_cat], axis=1)
cov_cols = list(cov_df.columns)
cov_df.head(5)

,age_corrected,race_Asian or Pacific Islander,race_Caucasian,race_Other
sample_name,,,,
1000565.0,50,1.0,0.0,0.0
1004472.0,41,0.0,1.0,0.0
1005221.0,66,0.0,1.0,0.0
1006556.0,56,0.0,1.0,0.0
1006792.0,65,0.0,1.0,0.0
1006976.0,61,0.0,1.0,0.0
1008913.0,45,0.0,1.0,0.0
1009912.0,52,0.0,1.0,0.0
1010562.0,57,0.0,1.0,0.0


In [3]:
# 2.5) ROW FILTERING BY MISSINGNESS
# - Decide which columns count toward per-row missingness
# - Optionally "skip" very sparse columns so they don't nuke N
# - Enforces outcome + covariates are present for regression

# Columns to IGNORE in the row-missingness calculation
# (edit this list and re-run this cell to see N change)
skip_cols = [
    "sugar_sweetened_drink_frequency",
    "free_sugar_scaled_0_5",
    "artificial_sweeteners",
    "one_liter_of_water_a_day_frequency",
    "olive_oil",
    "prepared_meals_frequency",
    "ready_to_eat_meals_frequency",
    "probiotic_frequency",
    "whole_eggs",
    "vitamin_b_supplement_frequency",
    "vitamin_d_supplement_frequency",
    "sugary_sweets_frequency"
]

max_missing_frac = 0.10  # 10% missing allowed (i.e., >=90% complete)
required_for_regression = [cognition_col] + cov_num # these have to be present, otherwise, drop the row

def filter_rows_by_missingness(df, *, skip_cols=(), required_cols=(), max_missing_frac=0.10):
    # if a row has 10 percent of the data missing, keep it
    skip_cols = list(skip_cols)
    required_cols = list(required_cols)
    # warn if skip columns aren't present
    missing_skip = [c for c in skip_cols if c not in df.columns]
    if missing_skip:
        print(f"[WARN] skip_cols not in dataframe (ignored): {missing_skip[:10]}{'...' if len(missing_skip)>10 else ''}")
    # columns that count toward per-row missingness
    cols_check = [c for c in df.columns if c not in set(skip_cols)]
    if len(cols_check) == 0:
        raise ValueError("After applying skip_cols, cols_check is empty. Reduce skip_cols.")
    missing_frac = df[cols_check].isna().mean(axis=1)
    n_in = df.shape[0]
    mask = missing_frac <= max_missing_frac
    df_f = df.loc[mask].copy()
    # enforce outcome/covariates present
    required_present = [c for c in required_cols if c in df_f.columns]
    df_f = df_f.dropna(subset=required_present)
    n_out = df_f.shape[0]
    report = {
        "n_in": n_in,
        "n_out": n_out,
        "dropped": n_in - n_out,
        "cols_check_n": len(cols_check),
        "missing_frac_summary": missing_frac.describe().to_dict()
    }
    return df_f, report

ukb_df, miss_report = filter_rows_by_missingness(
    ukb_df,
    skip_cols=skip_cols,
    required_cols=required_for_regression,
    max_missing_frac=max_missing_frac
)

print("\n[ROW FILTER REPORT]")
print(f"Input rows:   {miss_report['n_in']}")
print(f"Output rows:  {miss_report['n_out']}")
print(f"Dropped rows: {miss_report['dropped']}")
print(f"Cols counted toward missingness: {miss_report['cols_check_n']}")
print("Missingness fraction summary (before filtering):")
print(pd.Series(miss_report["missing_frac_summary"]))

[WARN] skip_cols not in dataframe (ignored): ['sugary_sweets_frequency']

[ROW FILTER REPORT]
Input rows:   502244
Output rows:  160568
Dropped rows: 341676
Cols counted toward missingness: 37
Missingness fraction summary (before filtering):
count    502244.000000
mean          0.070646
std           0.035937
min           0.000000
25%           0.054054
50%           0.081081
75%           0.081081
max           0.621622
dtype: float64


In [4]:
# 3) BUILD UKB X MATRIX WITH EXACT TRAINING COLUMNS
#    - missing columns become NaN (so imputer can handle)
#    - strings are coerced to NaN

X_ukb = ukb_df.reindex(columns=train_cols) # Training column comes fromt the AGP. if UKB has a training column, include it.
# if ukb does not have that training colunn, that column is created and filled with NaN
X_ukb = X_ukb.apply(pd.to_numeric, errors="coerce") # Coerce to numeric: any strings like "United Kingdom" -> NaN

# Report how many entire columns are missing from UKB
missing_cols = [c for c in train_cols if c not in ukb_df.columns]
print(f"UKB is missing {len(missing_cols)} / {len(train_cols)} training columns.")
if len(missing_cols) > 0:
    print("First 25 missing training columns:", missing_cols[:25])

# Report overall missingness in the matrix (after coercion)
missing_rate = float(np.mean(pd.isna(X_ukb.values)))
print(f"Overall missing rate in X_ukb (after numeric coercion): {missing_rate*100:.2f}%")

# 4) PREDICT PC_hat FOR UKB|
pc_hat = pd.DataFrame(index=ukb_df.index)

for k in range(n_pcs_to_predict):
    pc_name = f"PC{k+1}"
    if pc_name not in final_rf_models:
        raise KeyError(f"Model for {pc_name} not found in final_rf_models.pkl (keys={list(final_rf_models.keys())[:5]}...)")
    pc_hat[f"{pc_name}_hat"] = final_rf_models[pc_name].predict(X_ukb)

pc_hat_path = os.path.join(out_dir, "ukb_predicted_microbiome_pcs.csv")
pc_hat.to_csv(pc_hat_path)
print(f"Saved predicted PCs to: {pc_hat_path}")

UKB is missing 0 / 29 training columns.
Overall missing rate in X_ukb (after numeric coercion): 18.51%
Saved predicted PCs to: ./hongrui_results/UKB_RF_PCA_Projection/ukb_predicted_microbiome_pcs.csv


In [5]:
# ============================================================
# 5) OLS REGRESSION (HC3): cognition ~ PC_hat + covariates
# - age_corrected treated as CONTINUOUS
# - race + country_of_birth treated as CATEGORICAL (one-hot)
# - uses a single dropna() so sample is consistent
# ============================================================

# Outcome
y = pd.to_numeric(ukb_df[cognition_col], errors="coerce")
pc_cols = [c for c in pc_hat.columns if c.endswith("_hat")]
pc_hat_num = pc_hat[pc_cols].apply(pd.to_numeric, errors="coerce") # make sure predicted PC is numerical

X_reg = pd.concat([pc_hat_num, cov_df], axis=1) # full regression matrix
X_reg = sm.add_constant(X_reg, has_constant="add") # add intercept for the sm.OLS model
X_reg = X_reg.replace([np.inf, -np.inf], np.nan)
X_reg = X_reg.apply(pd.to_numeric, errors="coerce") # make sure everything is numeric

# Single analysis dataframe (locks sample)
data = pd.concat([y.rename("y"), X_reg], axis=1).dropna()

print("Rows used in regression:", data.shape[0], "Cols:", data.shape[1])
if data.shape[0] == 0:
    raise ValueError("No rows left after dropna. Check missingness in y / PCs / covariates.")

y_clean = data["y"].astype(float)
X_clean = data.drop(columns=["y"]).astype(float)

fit = sm.OLS(y_clean, X_clean).fit(cov_type="HC3")
print(fit.summary())

summary_path = os.path.join(out_dir, "ukb_cognition_on_predicted_pcs_summary.txt")
with open(summary_path, "w") as f:
    f.write(fit.summary().as_text())
print(f"Saved regression summary to: {summary_path}")

Rows used in regression: 160568 Cols: 16
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.065
Model:                            OLS   Adj. R-squared:                  0.065
Method:                 Least Squares   F-statistic:                     986.1
Date:                Fri, 27 Feb 2026   Prob (F-statistic):               0.00
Time:                        14:31:54   Log-Likelihood:            -3.4497e+05
No. Observations:              160568   AIC:                         6.900e+05
Df Residuals:                  160553   BIC:                         6.901e+05
Df Model:                          14                                         
Covariance Type:                  HC3                                         
                                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------

In [6]:
'''
tmp = pd.concat(
    [ukb_df[[cognition_col]].rename(columns={cognition_col:"y"}),
     ukb_df[["age_corrected"]]],
    axis=1
).apply(pd.to_numeric, errors="coerce").dropna()

fit_age_only = sm.OLS(tmp["y"], sm.add_constant(tmp[["age_corrected"]])).fit(cov_type="HC3")
print(fit_age_only.summary())
'''

'\ntmp = pd.concat(\n    [ukb_df[[cognition_col]].rename(columns={cognition_col:"y"}),\n     ukb_df[["age_corrected"]]],\n    axis=1\n).apply(pd.to_numeric, errors="coerce").dropna()\n\nfit_age_only = sm.OLS(tmp["y"], sm.add_constant(tmp[["age_corrected"]])).fit(cov_type="HC3")\nprint(fit_age_only.summary())\n'

In [7]:
print(ukb_df["age_corrected"].dtype)
print(ukb_df["age_corrected"].describe())
print("nunique:", ukb_df["age_corrected"].nunique())

int64
count    160568.000000
mean         56.733720
std           8.132477
min          39.000000
25%          50.000000
50%          58.000000
75%          63.000000
max          70.000000
Name: age_corrected, dtype: float64
nunique: 32


In [8]:
# 6) BASELINE MODEL (covariates only) — continuous age + categorical dummies
y_base = pd.to_numeric(ukb_df[cognition_col], errors="coerce")

X_base = sm.add_constant(cov_df, has_constant="add")
X_base = X_base.replace([np.inf, -np.inf], np.nan)
X_base = X_base.apply(pd.to_numeric, errors="coerce")

base_data = pd.concat([y_base.rename("y"), X_base], axis=1).dropna()
print("Rows used in BASELINE regression:", base_data.shape[0], "Cols:", base_data.shape[1])

fit_base = sm.OLS(
    base_data["y"].astype(float),
    base_data.drop(columns=["y"]).astype(float)
).fit(cov_type="HC3")

print(fit_base.summary())

Rows used in BASELINE regression: 160568 Cols: 6
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.053
Model:                            OLS   Adj. R-squared:                  0.053
Method:                 Least Squares   F-statistic:                     2889.
Date:                Fri, 27 Feb 2026   Prob (F-statistic):               0.00
Time:                        14:31:54   Log-Likelihood:            -3.4603e+05
No. Observations:              160568   AIC:                         6.921e+05
Df Residuals:                  160563   BIC:                         6.921e+05
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------

In [9]:
# ============================================================
# PERMUTATION TEST (Freedman–Lane): PCs add nothing beyond covariates
# - Fits baseline: y ~ covariates
# - Permutes baseline residuals
# - Re-fits full: y_perm ~ covariates + PC_hat
# - Tests joint Wald for PC terms + min single-PC p
# ============================================================

# config
B = 1000
seed = 42
rng = np.random.default_rng(seed)

pc_cols = [c for c in pc_hat.columns if c.endswith("_hat")]  # PC1_hat..PCk_hat

# ---- Build covariate dummies ONCE (must match your OLS regression encoding) ----
if len(reg_covars) > 0:
    cov_df = pd.get_dummies(
        ukb_df[reg_covars].astype("string").fillna("MISSING"),
        prefix=reg_covars,
        drop_first=True,
        dtype=float
    )
else:
    cov_df = pd.DataFrame(index=ukb_df.index)

# ---- Build one analysis dataframe (locks sample) ----
df = pd.concat(
    [
        ukb_df[[cognition_col]].rename(columns={cognition_col: "y"}),
        pc_hat[pc_cols].apply(pd.to_numeric, errors="coerce"),
        cov_df
    ],
    axis=1
).replace([np.inf, -np.inf], np.nan)

df["y"] = pd.to_numeric(df["y"], errors="coerce")
df = df.dropna()

y = df["y"].astype(float)

cov_cols = list(cov_df.columns)  # after get_dummies
X_base = sm.add_constant(df[cov_cols], has_constant="add") if cov_cols else sm.add_constant(pd.DataFrame(index=df.index), has_constant="add")
X_full = sm.add_constant(df[pc_cols + cov_cols], has_constant="add") if cov_cols else sm.add_constant(df[pc_cols], has_constant="add")

def fit_ols_hc3(yvec, Xmat):
    return sm.OLS(yvec, Xmat).fit(cov_type="HC3")

def joint_wald_stat(fit, cols_to_test):
    param_names = list(fit.params.index)
    R = np.zeros((len(cols_to_test), len(param_names)))
    for i, col in enumerate(cols_to_test):
        R[i, param_names.index(col)] = 1.0
    w = fit.wald_test(R, scalar=True)
    return float(w.statistic)

# ---- Observed (full) ----
fit_obs = fit_ols_hc3(y, X_full)
obs_joint = joint_wald_stat(fit_obs, pc_cols)
obs_pvals = fit_obs.pvalues[pc_cols].copy()
obs_minp = float(obs_pvals.min())

print("Observed R2 (full):", float(fit_obs.rsquared))
print("Observed joint Wald stat (PCs):", obs_joint)
print("Observed min single-PC p:", obs_minp)

# ---- Freedman–Lane permutations ----
# baseline fit: y ~ covariates
fit_base = fit_ols_hc3(y, X_base)
yhat = fit_base.fittedvalues
resid = y - yhat

perm_joint = np.empty(B)
perm_minp  = np.empty(B)
perm_sig_counts = {c: 0 for c in pc_cols}

resid_vals = resid.values

for b in range(B):
    resid_perm = rng.permutation(resid_vals)
    y_perm = yhat.values + resid_perm

    fit_b = fit_ols_hc3(y_perm, X_full)
    perm_joint[b] = joint_wald_stat(fit_b, pc_cols)

    pvals_b = fit_b.pvalues[pc_cols].values
    perm_minp[b] = float(np.min(pvals_b))

    for col, pv in zip(pc_cols, pvals_b):
        if pv < 0.05:
            perm_sig_counts[col] += 1

p_emp_joint = (1.0 + np.sum(perm_joint >= obs_joint)) / (B + 1.0)
p_emp_minp  = (1.0 + np.sum(perm_minp <= obs_minp)) / (B + 1.0)

print("\n--- Permutation results (Freedman–Lane) ---")
print(f"Permutations: {B}")
print(f"Empirical p (JOINT PCs | covariates): {p_emp_joint:.4g}")
print(f"Empirical p (MIN single-PC p): {p_emp_minp:.4g}")

sig_rates = pd.Series({col: perm_sig_counts[col] / B for col in pc_cols}).sort_values(ascending=False)
print("\nNull significance rate per PC at alpha=0.05 (should be ~0.05):")
print(sig_rates)

Observed R2 (full): 0.0709572539613188
Observed joint Wald stat (PCs): 2089.071319414263
Observed min single-PC p: 1.001315650303511e-133

--- Permutation results (Freedman–Lane) ---
Permutations: 1000
Empirical p (JOINT PCs | covariates): 0.000999
Empirical p (MIN single-PC p): 0.000999

Null significance rate per PC at alpha=0.05 (should be ~0.05):
PC2_hat     0.059
PC7_hat     0.055
PC4_hat     0.053
PC9_hat     0.051
PC10_hat    0.051
PC1_hat     0.045
PC3_hat     0.045
PC6_hat     0.045
PC8_hat     0.045
PC5_hat     0.041
dtype: float64


In [10]:
# Replace your current binary_outcomes block with this
binary_outcomes = [
    "ibs",
    "crohns_disease",
    "ulcerative_colitis",
    "dementia_general",
]

# --- Precompute predictors once (numeric PCs + encoded covariates) ---
pc_cols = [c for c in pc_hat.columns if c.endswith("_hat")]
pc_hat_num = pc_hat[pc_cols].apply(pd.to_numeric, errors="coerce")

# Build cov_df ONCE (continuous + categorical) — IMPORTANT: do NOT cast age to string
cov_parts = []

# numeric continuous covariates
if "cov_num" in globals() and len(cov_num) > 0:
    cov_num_df = ukb_df[cov_num].apply(pd.to_numeric, errors="coerce")
    cov_parts.append(cov_num_df)

# categorical covariates
if "cov_cat" in globals() and len(cov_cat) > 0:
    cov_cat_df = pd.get_dummies(
        ukb_df[cov_cat].astype("string").fillna("MISSING"),
        prefix=cov_cat,
        drop_first=True,
        dtype=float
    )
    cov_parts.append(cov_cat_df)

cov_df = pd.concat(cov_parts, axis=1) if cov_parts else pd.DataFrame(index=ukb_df.index)
cov_cols = list(cov_df.columns)

def joint_wald_pvalue(fit, cols_to_test):
    """H0: all coefficients of cols_to_test are 0 (joint Wald test)."""
    param_names = list(fit.params.index)
    R = np.zeros((len(cols_to_test), len(param_names)))
    for i, col in enumerate(cols_to_test):
        if col in param_names:
            R[i, param_names.index(col)] = 1.0
    w = fit.wald_test(R, scalar=True)
    return float(w.pvalue)

def fit_logit_with_fallback(y, X):
    """
    Try standard Logit + HC3 robust SE first.
    If that fails, try L1 regularized with alpha=0.0, then alpha=0.1.

    Returns
    -------
    fit_result
    fit_method : str
        One of:
        - "standard_logit_hc3"
        - "l1_regularized"
    alpha_used : float
        np.nan for standard fit, otherwise the alpha used in fit_regularized
    failure_reason : str or None
        Error message from the standard fit path if it failed
    """
    try:
        res = sm.Logit(y, X).fit(disp=0, maxiter=200)
        res_rob = res.get_robustcov_results(cov_type="HC3")
        return res_rob, "standard_logit_hc3", np.nan, None

    except Exception as e1:
        failure_reason = repr(e1)

        try:
            res_reg = sm.Logit(y, X).fit_regularized(
                method="l1",
                alpha=0.0,
                disp=0
            )
            return res_reg, "l1_regularized", 0.0, failure_reason

        except Exception as e2:
            failure_reason = failure_reason + " || alpha_0_failed: " + repr(e2)

            res_reg = sm.Logit(y, X).fit_regularized(
                method="l1",
                alpha=0.1,
                disp=0
            )
            return res_reg, "l1_regularized", 0.1, failure_reason

def coerce_binary(series):
    """Return 0/1 int series or None if not binary."""
    s = pd.to_numeric(series, errors="coerce")
    s2 = s.dropna()
    uniq = sorted(s2.unique().tolist())
    if len(uniq) == 0:
        return None
    if set(uniq).issubset({0, 1}):
        return s.astype("Int64")  # allow NA
    if set(uniq).issubset({1, 2}):
        return (s - 1).astype("Int64")
    return None

# --- Hard Code Dementia ---
dementia_subtypes = [
    "dementia_alzheimers",
    "dementia_vascular",
    "dementia_other",
    "dementia_unspecified"
]

missing_dementia_cols = [c for c in dementia_subtypes if c not in ukb_df.columns]
if missing_dementia_cols:
    print(f"[WARN] Missing dementia subtype columns: {missing_dementia_cols}")

present_dementia_cols = [c for c in dementia_subtypes if c in ukb_df.columns]
if len(present_dementia_cols) == 0:
    raise ValueError("No dementia subtype columns found in ukb_df.")

# Coerce each subtype to 0/1 first
dementia_bin_parts = {}
for c in present_dementia_cols:
    s = coerce_binary(ukb_df[c])
    if s is None:
        raise ValueError(f"{c} is not binary-coded as 0/1 or 1/2.")
    dementia_bin_parts[c] = s

dementia_bin_df = pd.DataFrame(dementia_bin_parts, index=ukb_df.index)

# Union across subtype columns:
# 1 if ANY dementia subtype is positive, else 0
# Do NOT sum, because the same person could have more than one subtype flag
ukb_df["dementia_general"] = dementia_bin_df.fillna(0).max(axis=1).astype("Int64")

# If all subtype fields are missing for a row, keep the combined outcome as missing
all_dem_missing = dementia_bin_df.isna().all(axis=1)
ukb_df.loc[all_dem_missing, "dementia_general"] = pd.NA

print("\nCombined dementia_general counts:")
print(ukb_df["dementia_general"].value_counts(dropna=False))

logit_outcome_rows = []
logit_pc_rows = []

for outcome in binary_outcomes:
    if outcome not in ukb_df.columns:
        print(f"[SKIP] {outcome} not found in ukb_df columns.")
        continue

    # Build modeling dataframe using precomputed predictors
    df_log = pd.concat(
        [
            ukb_df[[outcome]].rename(columns={outcome: "y"}),
            pc_hat_num,
            cov_df
        ],
        axis=1
    ).replace([np.inf, -np.inf], np.nan)

    # Coerce y to binary
    y_bin = coerce_binary(df_log["y"])
    if y_bin is None:
        print(f"[SKIP] {outcome}: not binary-coded (expected 0/1 or 1/2).")
        continue
    df_log["y"] = y_bin

    # Drop missing rows (locks sample for this outcome)
    needed = ["y"] + pc_cols + cov_cols
    df_log = df_log.dropna(subset=needed)
    if df_log.shape[0] == 0:
        print(f"[SKIP] {outcome}: no complete rows after dropna.")
        continue

    y_vec = df_log["y"].astype(int)
    n = int(len(y_vec))
    n_cases = int(y_vec.sum())
    n_controls = n - n_cases

    if n_cases == 0 or n_controls == 0:
        print(f"[SKIP] {outcome}: degenerate (cases={n_cases}, controls={n_controls}).")
        continue

    # Design matrix: PCs + covariates
    Xmat = df_log[pc_cols + cov_cols].astype(float)
    Xmat = sm.add_constant(Xmat, has_constant="add")

    # Fit
    fit_res, fit_method, alpha_used, failure_reason = fit_logit_with_fallback(y_vec, Xmat)
    used_reg = (fit_method != "standard_logit_hc3")

    # Joint PC test
    try:
        p_joint = joint_wald_pvalue(fit_res, pc_cols)
    except Exception:
        p_joint = np.nan

    logit_outcome_rows.append({
        "outcome": outcome,
        "n": n,
        "cases": n_cases,
        "controls": n_controls,
        "used_regularized_fit": used_reg,
        "fit_method": fit_method,
        "alpha_used": alpha_used,
        "failure_reason": failure_reason,
        "p_joint_all_PCs": p_joint
    })

    # Per-PC outputs
    if hasattr(fit_res, "params"):
        for pc in pc_cols:
            if pc in fit_res.params.index:
                logit_pc_rows.append({
                    "outcome": outcome,
                    "predictor": pc,
                    "coef_log_odds": float(fit_res.params[pc]),
                    "p_value": float(fit_res.pvalues[pc]) if hasattr(fit_res, "pvalues") else np.nan
                })

# Convert to DataFrames
outcome_df = pd.DataFrame(logit_outcome_rows)
pc_df = pd.DataFrame(logit_pc_rows)

# Multiple-testing correction across outcomes (joint tests)
if outcome_df.shape[0] > 0:
    mask = outcome_df["p_joint_all_PCs"].notna()
    pvals = outcome_df.loc[mask, "p_joint_all_PCs"].values
    rej, qvals, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")
    outcome_df.loc[mask, "q_joint_all_PCs_BH"] = qvals
    outcome_df.loc[mask, "reject_FDR_0p05_joint"] = rej

# Optional: BH across ALL per-PC tests (outcome x PC)
if pc_df.shape[0] > 0 and pc_df["p_value"].notna().any():
    mask2 = pc_df["p_value"].notna()
    pvals2 = pc_df.loc[mask2, "p_value"].values
    rej2, qvals2, _, _ = multipletests(pvals2, alpha=0.05, method="fdr_bh")
    pc_df.loc[mask2, "q_value_BH"] = qvals2
    pc_df.loc[mask2, "reject_FDR_0p05"] = rej2

# Save results
logit_outcome_path = os.path.join(out_dir, "ukb_logistic_outcomes_joint_tests.csv")
logit_pc_path = os.path.join(out_dir, "ukb_logistic_pc_level_results.csv")

outcome_df.sort_values("p_joint_all_PCs").to_csv(logit_outcome_path, index=False)
pc_df.to_csv(logit_pc_path, index=False)

print("\n=== Logistic regression (PC_hat) complete ===")
print("Saved outcome-level joint tests to:", logit_outcome_path)
print("Saved PC-level results to:", logit_pc_path)
print("\nTop outcomes by joint p-value:")
print(outcome_df.sort_values("p_joint_all_PCs").head(10))


Combined dementia_general counts:
dementia_general
0    158458
1      2110
Name: count, dtype: Int64


/scratch/liuhon33/parallel/AGPMicrobiomeHostPredictions/my_sklearn_env/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/scratch/liuhon33/parallel/AGPMicrobiomeHostPredictions/my_sklearn_env/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/scratch/liuhon33/parallel/AGPMicrobiomeHostPredictions/my_sklearn_env/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/scratch/liuhon33/parallel/AGPMicrobiomeHostPredictions/my_sklearn_env/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/scratch/liuhon33/parallel/AGPMicrobiomeHostPredictions/my_sklearn_env/lib/python3.11/


=== Logistic regression (PC_hat) complete ===
Saved outcome-level joint tests to: ./hongrui_results/UKB_RF_PCA_Projection/ukb_logistic_outcomes_joint_tests.csv
Saved PC-level results to: ./hongrui_results/UKB_RF_PCA_Projection/ukb_logistic_pc_level_results.csv

Top outcomes by joint p-value:
              outcome       n  cases  controls  used_regularized_fit  \
0                 ibs  160568  11355    149213                  True   
2  ulcerative_colitis  160568   2099    158469                  True   
1      crohns_disease  160568   1051    159517                  True   
3    dementia_general  160568   2110    158458                  True   

       fit_method  alpha_used  \
0  l1_regularized         0.0   
2  l1_regularized         0.0   
1  l1_regularized         0.0   
3  l1_regularized         0.0   

                                      failure_reason  p_joint_all_PCs  \
0  AttributeError("'LogitResults' object has no a...    3.374634e-296   
2  AttributeError("'LogitResults'